In [ ]:
# CELL 1: imports, paths, config

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
DATA_CLEAN = PROJECT_ROOT / "00_data" / "clean"
DATA_PROC = PROJECT_ROOT / "00_data" / "processed"
DATA_OUT = PROJECT_ROOT / "00_data" / "processed"

DATA_OUT.mkdir(parents=True, exist_ok=True)

# States we keep (USPS abbreviations)
KEEP_STATES = ["FL", "GA", "AL", "SC", "WA", "CO", "ID", "OR"]

# Treatment / control sets
TREATED_FL = ["FL"]
CONTROL_FL = ["GA", "AL", "SC"]

TREATED_WA = ["WA"]
CONTROL_WA = ["CO", "ID", "OR"]


def std_state_abbrev(series: pd.Series) -> pd.Series:
    """Uppercase and map full names to USPS abbreviations if necessary."""
    s = series.astype(str).str.strip().str.upper()

    name_to_abbrev = {
        "FLORIDA": "FL",
        "GEORGIA": "GA",
        "ALABAMA": "AL",
        "SOUTH CAROLINA": "SC",
        "WASHINGTON": "WA",
        "COLORADO": "CO",
        "IDAHO": "ID",
        "OREGON": "OR",
    }

    return s.replace(name_to_abbrev)


def std_county(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper()

In [ ]:
# CELL 2: Load county population file (wide format) and convert to long format

pop_path = DATA_CLEAN / "county_population_2000_2015.csv"
pop_raw = pd.read_csv(pop_path)

print("Population columns:", pop_raw.columns)

# ---------------------------
# Rename important columns
# ---------------------------
pop_raw = pop_raw.rename(
    columns={"STATE_ABBR": "state", "CTYNAME": "county", "FIPS": "fips"}
)

# Standardize strings
pop_raw["state"] = std_state_abbrev(pop_raw["state"])
pop_raw["county"] = std_county(pop_raw["county"])

# ---------------------------
# Identify population columns (POPESTIMATEYYYY)
# ---------------------------
pop_cols = [c for c in pop_raw.columns if c.startswith("POPESTIMATE")]
print("Population year columns:", pop_cols)

# Extract year integers from column names
# POPESTIMATE2001 → 2001
year_map = {c: int(c.replace("POPESTIMATE", "")) for c in pop_cols}

# ---------------------------
# Melt wide → long
# ---------------------------
pop = pop_raw.melt(
    id_vars=["state", "county", "fips", "STATE_FIPS", "COUNTY_FIPS", "STNAME"],
    value_vars=pop_cols,
    var_name="year_col",
    value_name="population",
)

# Map year_col → integer year
pop["year"] = pop["year_col"].map(year_map)

# Drop helper
pop = pop.drop(columns=["year_col"])

# Keep only states of interest
pop = pop[pop["state"].isin(KEEP_STATES)].copy()

# Drop rows with missing population
pop = pop.dropna(subset=["population"])

print("Long-format population shape:", pop.shape)
pop.head()

Population columns: Index(['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME',
       'ESTIMATESBASE2000', 'POPESTIMATE2001', 'POPESTIMATE2002',
       'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005',
       'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008',
       'POPESTIMATE2009', 'CENSUS2010POP', 'POPESTIMATE2010', 'STATE_FIPS',
       'COUNTY_FIPS', 'FIPS', 'POPESTIMATE2011', 'POPESTIMATE2012',
       'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015', 'STATE_ABBR'],
      dtype='object')
Population year columns: ['POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'POPESTIMATE2010', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015']
Long-format population shape: (7830, 8)


,state,county,fips,STATE_FIPS,COUNTY_FIPS,STNAME,population,year
0,AL,AUTAUGA COUNTY,1001,1,1,Alabama,44889,2001
1,AL,BALDWIN COUNTY,1003,1,3,Alabama,144875,2001
2,AL,BARBOUR COUNTY,1005,1,5,Alabama,28863,2001
3,AL,BIBB COUNTY,1007,1,7,Alabama,21028,2001
4,AL,BLOUNT COUNTY,1009,1,9,Alabama,51845,2001


In [ ]:
# CELL 3: load ARCOS county-year

arcos_path = DATA_PROC / "arcos_county_year.csv"
arcos = pd.read_csv(arcos_path)

print("ARCOS columns:", arcos.columns)

# Expecting: state, county, year, total_pills, total_mme, n_transactions
# If your names differ, adjust here:
rename_map_arcos = {}
if "state" in arcos.columns:
    rename_map_arcos["state"] = "state"
if "county" in arcos.columns:
    rename_map_arcos["county"] = "county"
if "year" in arcos.columns:
    rename_map_arcos["year"] = "year"
if "total_pills" in arcos.columns:
    rename_map_arcos["total_pills"] = "total_pills"
if "total_mme" in arcos.columns:
    rename_map_arcos["total_mme"] = "total_mme"
if "n_transactions" in arcos.columns:
    rename_map_arcos["n_transactions"] = "n_transactions"

arcos = arcos.rename(columns=rename_map_arcos)

arcos["state"] = std_state_abbrev(arcos["state"])
arcos["county"] = std_county(arcos["county"])
arcos["year"] = arcos["year"].astype(int)

arcos = arcos[arcos["state"].isin(KEEP_STATES)].copy()

print("ARCOS shape after filter:", arcos.shape)
arcos.head()

ARCOS columns: Index(['state', 'county', 'year', 'total_pills', 'total_mme',
       'n_transactions'],
      dtype='object')
ARCOS shape after filter: (7124, 6)


,state,county,year,total_pills,total_mme,n_transactions
330,AL,AUTAUGA,2006,2850573.246,5.166274e+07,8838
331,AL,AUTAUGA,2007,3167354.666,5.309262e+07,9760
332,AL,AUTAUGA,2008,3453070.648,5.746818e+07,10389
333,AL,AUTAUGA,2009,3759035.404,5.830612e+07,10528
334,AL,AUTAUGA,2010,3841098.472,6.263222e+07,11549


In [ ]:
# CELL 4: load mortality data (df_overdose_cleaned.csv)

mort_path = DATA_CLEAN / "df_overdose_cleaned.csv"
mort = pd.read_csv(mort_path)

print("Mortality columns:", mort.columns)

# --- rename to standard names ---

rename_map_mort = {
    "state": "state",
    "county": "county",
    "year": "year",
    "deaths": "drug_deaths",  # 👈 this is your key column
    "county_code": "fips_partial",  # optional, may be 3-digit county code
}

mort = mort.rename(columns=rename_map_mort)

# standardize fields
mort["state"] = std_state_abbrev(mort["state"])
mort["county"] = std_county(mort["county"])
mort["year"] = mort["year"].astype(int)

# keep only states of interest
mort = mort[mort["state"].isin(KEEP_STATES)].copy()

print("Mortality shape after filter:", mort.shape)
mort.head()

Mortality columns: Index(['county', 'county_code', 'year', 'drug/alcohol_induced_cause',
       'drug/alcohol_induced_cause_code', 'deaths', 'state'],
      dtype='object')
Mortality shape after filter: (1281, 7)


,county,fips_partial,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,drug_deaths,state
0,BALDWIN COUNTY,1003,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,29.0,AL
1,CALHOUN COUNTY,1015,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,12.0,AL
2,ETOWAH COUNTY,1055,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,AL
3,HOUSTON COUNTY,1069,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,10.0,AL
4,JEFFERSON COUNTY,1073,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,80.0,AL


In [ ]:
# CELL 5: build base panel from population and merge ARCOS + mortality

# restrict years to overlap across all three datasets
min_year = max(pop["year"].min(), arcos["year"].min(), mort["year"].min())
max_year = min(pop["year"].max(), arcos["year"].max(), mort["year"].max())

print("Using years:", min_year, "to", max_year)

panel = pop[(pop["year"] >= min_year) & (pop["year"] <= max_year)].copy()
print("Base panel (population) shape:", panel.shape)

# Merge ARCOS (left join so pop structure is preserved)
panel = panel.merge(
    arcos[["state", "county", "year", "total_pills", "total_mme", "n_transactions"]],
    on=["state", "county", "year"],
    how="left",
)

print("After merging ARCOS:", panel.shape)

# Merge mortality
panel = panel.merge(
    mort[["state", "county", "year", "drug_deaths"]],
    on=["state", "county", "year"],
    how="left",
)

print("After merging mortality:", panel.shape)
panel.head()

Using years: 2006 to 2015
Base panel (population) shape: (5220, 8)
After merging ARCOS: (5220, 11)
After merging mortality: (5220, 12)


,state,county,fips,STATE_FIPS,COUNTY_FIPS,STNAME,population,year,total_pills,total_mme,n_transactions,drug_deaths
0,AL,AUTAUGA COUNTY,1001,1,1,Alabama,51328,2006,NaN,NaN,NaN,NaN
1,AL,BALDWIN COUNTY,1003,1,3,Alabama,168121,2006,NaN,NaN,NaN,11.0
2,AL,BARBOUR COUNTY,1005,1,5,Alabama,27861,2006,NaN,NaN,NaN,NaN
3,AL,BIBB COUNTY,1007,1,7,Alabama,22099,2006,NaN,NaN,NaN,NaN
4,AL,BLOUNT COUNTY,1009,1,9,Alabama,55485,2006,NaN,NaN,NaN,NaN


In [ ]:
# CELL 6: create derived variables

# per-capita prescription measures
panel["mme_per_capita"] = panel["total_mme"] / panel["population"]
panel["pills_per_capita"] = panel["total_pills"] / panel["population"]

# overdose deaths per 100k (will be NaN where drug_deaths is missing / suppressed)
panel["overdoses_per_100k"] = panel["drug_deaths"] / panel["population"] * 100_000

# treat/control markers
panel["is_FL"] = (panel["state"] == "FL").astype(int)
panel["is_WA"] = (panel["state"] == "WA").astype(int)

panel["group_FL"] = np.where(
    panel["state"].isin(TREATED_FL),
    "treated_FL",
    np.where(panel["state"].isin(CONTROL_FL), "control_FL", "other"),
)

panel["group_WA"] = np.where(
    panel["state"].isin(TREATED_WA),
    "treated_WA",
    np.where(panel["state"].isin(CONTROL_WA), "control_WA", "other"),
)

panel["is_treated_state"] = panel["state"].isin(["FL", "WA"]).astype(int)

# post-policy indicators
panel["post_FL"] = (
    (panel["state"].isin(TREATED_FL + CONTROL_FL)) & (panel["year"] >= 2010)
).astype(int)
panel["post_WA"] = (
    (panel["state"].isin(TREATED_WA + CONTROL_WA)) & (panel["year"] >= 2012)
).astype(int)

panel["post"] = 0
panel.loc[(panel["state"] == "FL") & (panel["year"] >= 2010), "post"] = 1
panel.loc[(panel["state"] == "WA") & (panel["year"] >= 2012), "post"] = 1

panel[["state", "county", "year", "mme_per_capita", "overdoses_per_100k"]].head()

,state,county,year,mme_per_capita,overdoses_per_100k
0,AL,AUTAUGA COUNTY,2006,NaN,NaN
1,AL,BALDWIN COUNTY,2006,NaN,6.542907
2,AL,BARBOUR COUNTY,2006,NaN,NaN
3,AL,BIBB COUNTY,2006,NaN,NaN
4,AL,BLOUNT COUNTY,2006,NaN,NaN


In [ ]:
# CELL 7: save master panel with pop + ARCOS + mortality

out_csv = DATA_OUT / "master_county_year_panel.csv"
out_parquet = DATA_OUT / "master_county_year_panel.parquet"

panel.to_csv(out_csv, index=False)
panel.to_parquet(out_parquet, index=False)

print("Saved master panel to:")
print(" -", out_csv)
print(" -", out_parquet)

Saved master panel to:
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/master_county_year_panel.csv
 - /Volumes/T7/Opiod Project/pds-2025-opioids-straight-as/00_data/processed/master_county_year_panel.parquet


In [ ]:
# CELL 6: FEATURE ENGINEERING

import numpy as np

# -----------------------------
# 1. PER-CAPITA METRICS
# -----------------------------

panel["mme_per_capita"] = panel["total_mme"] / panel["population"]
panel["pills_per_capita"] = panel["total_pills"] / panel["population"]
panel["transactions_per_capita"] = panel["n_transactions"] / panel["population"]

panel["overdoses_per_100k"] = panel["drug_deaths"] / panel["population"] * 100_000
panel["pills_per_100"] = panel["total_pills"] / panel["population"] * 100
panel["mme_per_100"] = panel["total_mme"] / panel["population"] * 100

# -----------------------------
# 2. LOG TRANSFORMS
# -----------------------------
# add +1 to avoid log(0)
panel["log_mme"] = np.log1p(panel["total_mme"])
panel["log_pills"] = np.log1p(panel["total_pills"])
panel["log_overdoses"] = np.log1p(panel["drug_deaths"])
panel["log_mme_per_capita"] = np.log1p(panel["mme_per_capita"])
panel["log_overdoses_per_100k"] = np.log1p(panel["overdoses_per_100k"])

# -----------------------------
# 3. TREATMENT INDICATORS
# -----------------------------

# Treated states
panel["treated"] = panel["state"].isin(["FL", "WA"]).astype(int)

# FL policy year 2010
# WA policy year 2012
panel["fl_policy_year"] = 2010
panel["wa_policy_year"] = 2012

# State-specific post indicators
panel["post_FL"] = ((panel["state"] == "FL") & (panel["year"] >= 2010)).astype(int)
panel["post_WA"] = ((panel["state"] == "WA") & (panel["year"] >= 2012)).astype(int)

# Combined post for pooled model
panel["post"] = np.where(
    ((panel["state"] == "FL") & (panel["year"] >= 2010))
    | ((panel["state"] == "WA") & (panel["year"] >= 2012)),
    1,
    0,
)

# -----------------------------
# 4. TIME RELATIVE TO POLICY (EVENT STUDY)
# -----------------------------
panel["t_minus_policy"] = np.where(
    panel["state"] == "FL",
    panel["year"] - 2010,
    np.where(panel["state"] == "WA", panel["year"] - 2012, np.nan),
)

# Useful categorical event-study variable:
panel["event_time"] = panel["t_minus_policy"]

# -----------------------------
# 5. FIXED EFFECT HELPERS
# -----------------------------

# County FIPS makes a perfect county FE key
if "fips" in panel.columns:
    panel["county_fe"] = panel["fips"]
else:
    # fallback: concatenate state + county
    panel["county_fe"] = panel["state"] + "_" + panel["county"]

panel["year_fe"] = panel["year"]

# -----------------------------
# 6. TREND VARIABLES
# -----------------------------
panel["time_index"] = panel["year"] - panel["year"].min()
panel["time_index_sq"] = panel["time_index"] ** 2

# -----------------------------
# 7. MISSINGNESS FLAGS
# -----------------------------
panel["missing_mme"] = panel["total_mme"].isna().astype(int)
panel["missing_deaths"] = panel["drug_deaths"].isna().astype(int)
panel["missing_pills"] = panel["total_pills"].isna().astype(int)

# -----------------------------
# 8. CONTROL GROUP LABELS
# -----------------------------

# Florida treated + control states
FL_TREATED = ["FL"]
FL_CONTROLS = ["GA", "AL", "SC"]

panel["fl_group"] = np.where(
    panel["state"].isin(FL_TREATED),
    "treated_FL",
    np.where(panel["state"].isin(FL_CONTROLS), "control_FL", "other"),
)

# Washington treated + control states
WA_TREATED = ["WA"]
WA_CONTROLS = ["CO", "ID", "OR"]

panel["wa_group"] = np.where(
    panel["state"].isin(WA_TREATED),
    "treated_WA",
    np.where(panel["state"].isin(WA_CONTROLS), "control_WA", "other"),
)

panel.head()

,state,county,fips,STATE_FIPS,COUNTY_FIPS,STNAME,population,year,total_pills,total_mme,...,event_time,county_fe,year_fe,time_index,time_index_sq,missing_mme,missing_deaths,missing_pills,fl_group,wa_group
0,AL,AUTAUGA COUNTY,1001,1,1,Alabama,51328,2006,NaN,NaN,...,NaN,1001,2006,0,0,1,1,1,control_FL,other
1,AL,BALDWIN COUNTY,1003,1,3,Alabama,168121,2006,NaN,NaN,...,NaN,1003,2006,0,0,1,0,1,control_FL,other
2,AL,BARBOUR COUNTY,1005,1,5,Alabama,27861,2006,NaN,NaN,...,NaN,1005,2006,0,0,1,1,1,control_FL,other
3,AL,BIBB COUNTY,1007,1,7,Alabama,22099,2006,NaN,NaN,...,NaN,1007,2006,0,0,1,1,1,control_FL,other
4,AL,BLOUNT COUNTY,1009,1,9,Alabama,55485,2006,NaN,NaN,...,NaN,1009,2006,0,0,1,1,1,control_FL,other
